In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### DATA LOADING AND UNDERSTANDING

In [2]:
file_path = "..\Data\Raw\online_retail_II.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

df.head()

<>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
C:\Users\Ade\AppData\Local\Temp\ipykernel_26604\712113259.py:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  file_path = "..\Data\Raw\online_retail_II.xlsx"


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Jumlah baris dan kolom pada dataset
print("Database shape : ", df.shape)

# List kolom pada dataset
print("\nColumns:")
for c in df.columns:
    print("-", c)

# Tipe data dari tiap kolom di dataset
print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))

# Check missing value di tiap kolom
print("\nMissing values per column:")
display(df.isnull().sum().to_frame("missing").sort_values("missing", ascending=False))
# Check duplikasi data di dataset
print("\nDuplicate rows:", int(df.duplicated().sum()))

Database shape :  (1067371, 8)

Columns:
- Invoice
- StockCode
- Description
- Quantity
- InvoiceDate
- Price
- Customer ID
- Country

Dtypes:


,dtype
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str



Missing values per column:


,missing
Customer ID,243007
Description,4382
StockCode,0
Invoice,0
Quantity,0
InvoiceDate,0
Price,0
Country,0



Duplicate rows: 34335


### DATA PREPROCESSING

In [4]:
missing_value = pd.DataFrame(
    {
        "missing_count": df.isnull().sum(),
        "missing_percentage": round(df.isnull().mean()*100,2)
    }
)
display(missing_value.sort_values("missing_percentage",ascending =False))

# i'm not dropping the missing value, because the missing value only in customer_id and description column. 
# so its still have a valuable transaction data in its row

,missing_count,missing_percentage
Customer ID,243007,22.77
Description,4382,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


In [5]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 34335


In [6]:
df = df.drop_duplicates().copy()

print("Dataset shape after cleaning:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

Dataset shape after cleaning: (1033036, 8)
Remaining duplicates: 0


In [7]:
# Cancellation invoices
cancelled = df[
    df["Invoice"].astype(str).str.startswith("C")
]

# Negative quantity records
returns = df[df["Quantity"] < 0]

# Zero or negative prices
invalid_prices = df[df["Price"] <= 0]

print("Cancelled invoice records:", len(cancelled))
print("Negative quantity records:", len(returns))
print("Non-positive price records:", len(invalid_prices))

Cancelled invoice records: 19104
Negative quantity records: 22496
Non-positive price records: 6019


In [8]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
df = df.rename(columns={
    "invoice": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "unitprice": "unit_price",
    "customerid": "customer_id"
})

In [9]:
display(df.dtypes.to_frame())

df.head(5)

,0
invoice_no,object
stock_code,object
description,object
quantity,int64
invoice_date,datetime64[us]
price,float64
customer_id,float64
country,str


,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [10]:
# Date conversion
df["invoice_date"] = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

# Preserve invoice and product codes as strings
df["invoice_no"] = df["invoice_no"].astype("string")
df["stock_code"] = df["stock_code"].astype("string")
df["country"] = df["country"].astype("string")

#strip all string/object column
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

# Customer ID: handle missing values without losing IDs
df["customer_id"] = (
    pd.to_numeric(df["customer_id"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df.info()

<class 'pandas.DataFrame'>
Index: 1033036 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice_no    1033036 non-null  string        
 1   stock_code    1033036 non-null  string        
 2   description   1028757 non-null  object        
 3   quantity      1033036 non-null  int64         
 4   invoice_date  1033036 non-null  datetime64[us]
 5   price         1033036 non-null  float64       
 6   customer_id   797885 non-null   string        
 7   country       1033036 non-null  string        
dtypes: datetime64[us](1), float64(1), int64(1), object(1), string(4)
memory usage: 70.9+ MB


In [11]:
display(df.dtypes.to_frame())

df.head(5)

,0
invoice_no,string
stock_code,string
description,object
quantity,int64
invoice_date,datetime64[us]
price,float64
customer_id,string
country,string


,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [12]:
df["revenue"] = df["quantity"] * df["price"]
df.head(5)

,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [13]:
df["is_cancelled"] = (
    df["invoice_no"].str.upper().str.startswith("C", na=False)
)


df["is_negative_quantity"] = df["quantity"] < 0
pd.crosstab(
    df["is_cancelled"],
    df["is_negative_quantity"],
    margins=True
)

is_negative_quantity,False,True,All
is_cancelled,,,
False,1010539,3393,1013932
True,1,19103,19104
All,1010540,22496,1033036


In [14]:
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue,is_cancelled,is_negative_quantity
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,False,False


In [15]:
df["year"] = df["invoice_date"].dt.year
df["month"] = df["invoice_date"].dt.month

df["year_month"] = (
    df["invoice_date"].dt.to_period("M").astype("string")
)

df["day_name"] = df["invoice_date"].dt.day_name()
df["hour"] = df["invoice_date"].dt.hour
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue,is_cancelled,is_negative_quantity,year,month,year_month,day_name,hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,False,False,2009,12,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,False,False,2009,12,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,False,False,2009,12,2009-12,Tuesday,7


In [16]:
gross_sales = df[
    (~df["is_cancelled"]) &
    (df["quantity"] > 0) &
    (df["price"] > 0) &
    (df["invoice_date"].notna())
].copy()
print('Gross Sales Shape :', gross_sales.shape)
gross_sales.head()

Gross Sales Shape : (1007913, 16)


,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue,is_cancelled,is_negative_quantity,year,month,year_month,day_name,hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,False,False,2009,12,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,False,False,2009,12,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,False,False,2009,12,2009-12,Tuesday,7


In [21]:
raw_transactions =df.copy()

valid_base = (
    df["invoice_date"].notna() &
    df["price"].gt(0)
)

sales_mask = (
    valid_base &
    (~df["is_cancelled"]) &
    (df["quantity"] > 0)
)

fact_sales = df.loc[sales_mask].copy()

In [22]:
adjustment_mask = (
    df["is_cancelled"] |
    df["is_negative_quantity"]
)

fact_adjustments = df.loc[adjustment_mask].copy()

fact_adjustments["is_financial_adjustment"] = (
    fact_adjustments["price"].gt(0) &
    fact_adjustments["invoice_date"].notna() &
    fact_adjustments["revenue"].lt(0)
)

In [23]:
gross_sales = fact_sales["revenue"].sum()
negative_adjustments = fact_adjustments.loc[
    fact_adjustments["is_financial_adjustment"],
    "revenue"
].sum()
print("Gross sales :", round(gross_sales,2))
print("negative sales :", round(negative_adjustments,2 ))


Gross sales : 20476260.45
negative sales : -1462424.18


In [24]:
net_sales = gross_sales + negative_adjustments
print("net sales", net_sales)

net sales 19013836.268


In [ ]:
# Making a sample processed dataset for GitHub Repositories
import os

output_dir = os.path.join("..", "Data", "Processed")
os.makedirs(output_dir, exist_ok=True)

for table_name, dataframe in {
    "raw_transactions": raw_transactions,
    "fact_sales": fact_sales,
    "fact_adjustments": fact_adjustments,
}.items():
    sample_path = os.path.join(output_dir, f"{table_name}_sample.csv")
    
    # Export 1,000 rows without the pandas row index
    dataframe.head(1000).to_csv(sample_path, index=False)
    
    size_kb = os.path.getsize(sample_path) / 1024
    print(f"Saved {sample_path} ({size_kb:.1f} KB)")

Saved ..\Data\Processed\raw_transactions_sample.csv (133.0 KB)
Saved ..\Data\Processed\fact_sales_sample.csv (132.7 KB)
Saved ..\Data\Processed\fact_adjustments_sample.csv (135.8 KB)


In [ ]:
# raw_transactions =df.copy()
# raw_transactions.to_csv("..\\Data\\Processed\\raw_transactions.csv", index=False)
# fact_sales.to_csv("..\\Data\\Processed\\fact_sales.csv", index=False)
# fact_adjustments.to_csv("..\\Data\\Processed\\fact_adjustments.csv", index=False)

### Import Dataset into SQL Server Database


In [ ]:
# from sqlalchemy import create_engine, text
# from urllib.parse import quote_plus



# server = "localhost"  # e.g. localhost or SERVER\\INSTANCE
# database = "online_retail"
# driver = "ODBC Driver 18 for SQL Server"  # Must be installed on this machine

# odbc_connection = (
#     f"DRIVER={{{driver}}};"
#     f"SERVER={server};"
#     f"DATABASE={database};"
#     "Trusted_Connection=yes;"
#     "TrustServerCertificate=yes;"
# )

# connection_url = (
#     "mssql+pyodbc:///?odbc_connect="
#     + quote_plus(odbc_connection)
# )

# engine = create_engine(connection_url, fast_executemany=True)

# # Check the connection
# with engine.connect() as connection:
#     print(connection.execute(text("SELECT 1")).scalar())

# # Load the three dataframes into dbo tables
# for table_name, dataframe in {
#     "raw_transactions": raw_transactions,
#     "fact_sales": fact_sales,
#     "fact_adjustments": fact_adjustments,
# }.items():
#     dataframe.to_sql(
#         name=table_name,
#         con=engine,
#         schema="dbo",
#         if_exists="replace",  # Change to "append" to keep existing table data
#         index=False,
#         chunksize=5000,
#         method=None,
#     )
#     print(f"Loaded {len(dataframe):,} rows into dbo.{table_name}")

1
Loaded 1,033,036 rows into dbo.raw_transactions
Loaded 1,007,913 rows into dbo.fact_sales
Loaded 22,497 rows into dbo.fact_adjustments
